# AD COURSE MAP

## Data Analysis

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from leaspy import Leaspy, Data, AlgorithmSettings, IndividualParameters, __watermark__
from leaspy.io.logs.visualization.plotting import Plotting
from collections import defaultdict
import os
import time
from sklearn.model_selection import KFold
from scipy.stats import shapiro, probplot, t

In [7]:
# Read Dataset
dataset = pd.read_csv('../datasets/cognitive_scores.csv')
dataset['ID'] = dataset['ID'].astype('str') # Make id as a string

#definition of a sub group
sub_n = '1000' # number of subjects
base_path = f'outputs/experiment_{sub_n}_patients/' # BASE PATH for results
vis_n = dataset[dataset['ID']==sub_n].index[-1]+1 # number of visits (rows)
dataset = dataset.iloc[:vis_n]

In [8]:
# Create a folder to save the results
if not os.path.exists(base_path):
    os.makedirs(base_path)

text_file_name = base_path + "results.txt"
# Create a file
with open(text_file_name, "w") as file:
    file.write("Experiment with " + str(sub_n) + " patients")

In [9]:
# Function to append text to the file
def append_to_text_file(file_name, content):
    with open(file_name, "a") as file:
        file.write("\n" + content)

In [10]:
# Save number of visits
append_to_text_file(text_file_name, "N° Visite: " + str(vis_n))

In [ ]:
# Visit distribution
visit_distribution = dataset.groupby('ID').count()
visit_distribution = visit_distribution[['TIME']].rename(columns={'TIME':'N visits'})
visit_distribution.hist()
display(visit_distribution)

In [12]:
# Function to create a dictionary that collects subject IDs by number of visits
def split_sub_per_visits():
    sub_per_visits = []
    for i in range(visit_distribution['N visits'].max()+1):
        sub_per_visits.append([])

    # Populate sub_per_visits based on the number of visits for each subject
    for id in dataset['ID'].unique():
        i = int(id)
        n = visit_distribution.iloc[i][0]
        sub_per_visits[n].append(id)

    return sub_per_visits


# Function to split the dataset into train, validation
def split_dataset_in_train_val(dataset, percentage):
    # Create a dictionary that collects subject IDs by number of visits
    sub_per_visits = split_sub_per_visits()

    # Set seed for reproducibility and select IDs for training set.
    np.random.seed(0)

    train_id = []
    for n in range(len(sub_per_visits)):
        if sub_per_visits[n]:
            sublist_id = np.random.choice(sub_per_visits[n], size=int(percentage*len(sub_per_visits[n])), replace=False).tolist()
            train_id += sublist_id

    df_train = dataset[dataset['ID'].isin(train_id)]
    df_test = dataset[~dataset['ID'].isin(train_id)]

    # Get from the test dataset a dataset to predict 
    df_to_pred = df_test.groupby('ID').tail(1) # Get last visit of every patient

    list_index = df_to_pred.index.tolist()
    df_pers = df_test[~df_test.index.isin(list_index)] # Evaluation dataset

    return df_train, df_test, df_pers, df_to_pred
    

In [13]:
df_train, df_test, df_pers, df_to_pred = split_dataset_in_train_val(dataset, 0.8)

In [ ]:
for col in dataset.columns[3:]:
    max = dataset[col].max()
    min = dataset[col].min()
    print(col + ' interval = ' + str(min) + ' : ' + str(max))

In [15]:
df_train = df_train.set_index(['ID', 'TIME'])
df_pers = df_pers.set_index(['ID', 'TIME'])
df_to_pred = df_to_pred.set_index(['ID', 'TIME'])

In [16]:
# Creata Data Object for Leaspy
data_train = Data.from_dataframe(df_train)
data_pers = Data.from_dataframe(df_pers)
data_to_pred = Data.from_dataframe(df_to_pred)

## Train

### Model

In [18]:
# Import the JSON as a dictionary
import json

with open('algorithm_settings_calibration.json', "r") as file:
    json_data = json.load(file)

# Save training parameters
append_to_text_file(text_file_name, "Fit parameters: ")
append_to_text_file(text_file_name, "- N° iterazioni: " + str(json_data['parameters']['n_iter'])) # Number of iteration
append_to_text_file(text_file_name, "- N° burn in iterazioni: " + str(json_data['parameters']['n_burn_in_iter_frac'])) # Fraction of iterations burn in
append_to_text_file(text_file_name, "- N° burn in step power: " + str(json_data['parameters']['burn_in_step_power'])) # Step power of burn iterations in

In [ ]:
leaspy = Leaspy('logistic', 
                source_dimension=2,                  # number of dimensions for non-temporal inter-subject variability
                noise_model='gaussian_diagonal')     # estimate the residual noise scaling per feature
leaspy_plot = Plotting(leaspy.model)

In [ ]:
# Algorithm Settings
algo_settings = AlgorithmSettings.load('algorithm_settings_calibration.json') 

# Save logs
algo_settings.set_logs(
    path=base_path + '/logs', # Creates a logs file ; if existing, ask if rewrite it
    save_periodicity=50, # Saves the values in csv files every N iterations
    console_print_periodicity=100, # Displays logs in the console/terminal every N iterations, or None
    plot_periodicity=1000, # Generates the convergence plots every N iterations
    overwrite_logs_folder=True # if True and the logs folder already exists, it entirely overwrites it
)

### Train

In [ ]:
start_time = time.time() # Keeps track of training time
leaspy.fit(data_train, settings=algo_settings) # Fitting
end_time = time.time() # End training
execution_time = end_time - start_time # Keeps track of training time

In [22]:
# Save noise results
append_to_text_file(text_file_name, 'Model performance: ')
append_to_text_file(text_file_name, 'Time: ' + str(execution_time) + ' s')
for x, var in enumerate(df_train.columns):
    noise_std = round(leaspy.model.noise_model.to_dict()['scale'][x] * 100, 2)
    append_to_text_file(text_file_name, var + ':' + str(noise_std) + '%')

In [23]:
from datetime import datetime

now = datetime.now()
time = '_' +str(now.month) + '_' + str(now.day) + '-' + str(now.hour) + str(now.minute)

# Save model
leaspy.save(base_path + "model_parameters"+ time +".json")

In [ ]:
for file in os.listdir(base_path + 'logs\parameter_convergence'):
    filename = os.fsdecode(file)
    if filename.endswith('std.csv'):
        df = pd.read_csv(base_path + 'logs\\parameter_convergence\\' + filename, names=['index', 'std'])
        plt.plot(df[int(len(df['std']) * 0.8):]['std'])
        plt.title(filename)
        plt.show()

In [ ]:
# Create a folder for images
if not os.path.exists(base_path + '/images/'):
    os.makedirs(base_path + '/images/')

# Show population curves
ax = leaspy_plot.average_trajectory(alpha=1, figsize=(14,6), n_std_left=2, n_std_right=8)
plt.savefig(base_path + '/images/' + 'mean_curves.png')
plt.show()

In [26]:
from leaspy import IndividualParameters

# Save average parameters

mean_xi = leaspy.model.parameters['xi_mean'].numpy()
mean_tau = leaspy.model.parameters['tau_mean'].numpy()
mean_source = leaspy.model.parameters['sources_mean'].numpy().tolist()
number_of_sources = leaspy.model.source_dimension
mean_sources = [mean_source]*number_of_sources

average_parameters = {
    'xi': mean_xi,
    'tau': mean_tau,
    'sources': mean_sources
}

# Create an Individual Parameters Object to estimate biomarkers for a 'mean patient'
ip_average = IndividualParameters()
ip_average.add_individual_parameters('average', average_parameters)

timepoints = np.linspace(55, 130, 100)
values = leaspy.estimate({'average': timepoints}, ip_average)

In [ ]:
# Get MMSE estimated values
MMSE_mean = values['average'][:, 0] 

# Obtain the age range in which the MMSE is between 0.1 and 0.23 (23 and 27)
MCI_index = np.where((MMSE_mean >= 0.1) & (MMSE_mean<= 0.23 ))
ageMCI_a = timepoints[MCI_index[0][0]]
ageMCI_b = timepoints[MCI_index[0][-1]]

print('the mean population shifts to MCI (MMSE in [0.1 : 0.23]) at age: ' + str(round(ageMCI_a, 2)) + ' - ' + str(round(ageMCI_b, 2)))
append_to_text_file(text_file_name, "Mean population shifts to MCI: at age: " + str(round(ageMCI_a, 2)) + ' - ' + str(round(ageMCI_b, 2)))

In [24]:
append_to_text_file(text_file_name, "Personalization - with scipy_minimize")
settings_personalization = AlgorithmSettings('scipy_minimize', seed=0)

### Personalization

In [ ]:
# Personalize step
ip = leaspy.personalize(data_pers, settings_personalization)

In [ ]:
# Future predictions on biomarkers of patients who have had their biomarkers personalized
predictions = leaspy.estimate(dict(zip(df_to_pred.index.get_level_values('ID'),df_to_pred.index.get_level_values('TIME'))), ip)


# Create a dataframe from predictions
prediction_temp = {k:v[0] for k, v in predictions.items()} # Make a well format dictionary to majke a dataframe
df_predicted = pd.DataFrame.from_dict(prediction_temp, orient='index', columns=['MMSE', 'Memory', 'Language', 'Concentration', 'Praxis'])
display(df_predicted)

In [ ]:
append_to_text_file(text_file_name, "Prediction Mean absolute error: ")

mae = dict()
# Calculation of MAE for every biomarkers
for var in df_to_pred.columns:
    residuals = np.asarray(df_to_pred[var]) - np.asarray(df_predicted[var])
    error = np.abs(np.asarray(df_to_pred[var]) - np.asarray(df_predicted[var])).mean()

    # Supposed normal distribution due to the high volume of data
    #probplot(residuals, dist="norm", plot=plt)
    #plt.show()
    
    # To obtain theconfidence intervall we need:
    # - the std of the residuals
    res_std = np.std(residuals)
    # - length of the predicted data
    res_len = len(residuals)
    # - z for the 95% of coinfidence
    z = 1.96

    #confidece interval:
    intervals = z*res_std/np.sqrt(res_len)

    mae[var] = error
    print(var +' - MAE: ' + str(round(error, 4))+ ' ± ' + str(round(intervals,3)))
    append_to_text_file(text_file_name, var + ': ' + str(error))

In [ ]:
for x, var in enumerate(df_to_pred.columns):
    plt.plot(np.linspace(0, 1, 10), np.linspace(0, 1, 10), color='k')
    plt.scatter(df_to_pred[var], df_predicted[var])
    plt.title(var + ' - true value vs predicted')
    plt.xlabel('True value')
    plt.ylabel('Predicted value')
    plt.grid()
    plt.show()

## Crossvalidation: K-fold

In [29]:
def crossvalidation(dataset, k):
    kf = KFold(n_splits=k, shuffle=True, random_state=0)
    id_list = np.array(dataset['ID'].unique())
    k_mae = dict()

    for index_train, index_val in kf.split(id_list):
        id_train = id_list[index_train]
        id_val = id_list[index_val]
        df_train = dataset[dataset['ID'].isin(id_train)]
        df_val = dataset[dataset['ID'].isin(id_val)]
        df_to_pred = df_val.groupby('ID').tail(1)
        df_pers = df_val[~df_val.index.isin(df_to_pred.index.tolist())]

        #from df to data
        data_train = Data.from_dataframe(df_train.set_index(['ID', 'TIME']))
        data_pers = Data.from_dataframe(df_pers.set_index(['ID', 'TIME']))
        df_to_pred = df_to_pred.set_index(['ID', 'TIME'])

        leaspy = Leaspy('logistic', 
                source_dimension=2,                  # number of dimensions for non-temporal inter-subject variability
                noise_model='gaussian_diagonal')     # estimate the residual noise scaling per feature

        # Algorithm Settings
        algo_settings = AlgorithmSettings.load('algorithm_settings_calibration.json')
        # Fitting
        leaspy.fit(data_train, settings=algo_settings) 

        # Personalize step
        settings_personalization = AlgorithmSettings('scipy_minimize', seed=0)
        ip = leaspy.personalize(data_pers, settings_personalization)
        # Future predictions on biomarkers of patients who have had their biomarkers personalized
        predictions = leaspy.estimate(dict(zip(df_to_pred.index.get_level_values('ID'),df_to_pred.index.get_level_values('TIME'))), ip)

        # Create a dataframe from predictions
        prediction_temp = {k:v[0] for k, v in predictions.items()} # Make a well format dictionary to majke a dataframe
        df_predicted = pd.DataFrame.from_dict(prediction_temp, orient='index', columns=['MMSE', 'Memory', 'Language', 'Concentration', 'Praxis'])

        # Calculation of MAE for every biomarkers
        for x, var in enumerate(df_to_pred.columns):
            error = np.abs(np.asarray(df_to_pred[var]) - np.asarray(df_predicted[var])).mean()
            if var in k_mae.keys():
                k_mae[var].append(error)
            else:
                k_mae[var] = [error]

    return k_mae

In [ ]:
k_mae = crossvalidation(dataset, k=5)

In [ ]:
print('Mean kfold MAE vs. model prediction MAE')
for var in k_mae.keys():
    print(var+': '+ str(round(np.mean(k_mae[var]), 5)) + ' vs. ' + str(round(mae[var], 5)))

## Simulation 

In [ ]:
# Use customization and sample random effect variables from the data distribution
data_test = Data.from_dataframe(df_test.set_index(['ID', 'TIME']))

settings_ip_simulate = AlgorithmSettings('scipy_minimize', seed=0)
individual_params = leaspy.personalize(data_test, settings_ip_simulate)

In [ ]:
# Get number of visits, mean of number of visits and std of number of visits
n_visit = df_test.groupby('ID').count()[['TIME']].rename(columns={'TIME':'N visits'})
visit_mean = n_visit['N visits'].mean()
visit_std = n_visit['N visits'].std()
pred_sub = len(n_visit)

print('df_test numb. of visits, mean & std:' + str(visit_mean) + ' & ' + str(visit_std))
print('df_test numb. of subjects: ' + str(pred_sub))

In [ ]:
# Similuated data
settings_simulate = AlgorithmSettings('simulation', seed=0, number_of_subjects=pred_sub, mean_number_of_visits=visit_mean, std_number_of_visits=visit_std)
simulated_data = leaspy.simulate(individual_params, data_test, settings_simulate)

In [ ]:
# Make a dataframe of simulated data
df_simu = simulated_data.data.to_dataframe().set_index(['ID', 'TIME'])
display(df_simu)

In [ ]:
# Compare the datasets distribution (real biomarkers vs virtual biomarkers)
for var in df_simu.columns:
    plt.hist(df_test[var], alpha=0.5, label='original distribution')
    plt.hist(df_simu[var], alpha=0.5, label='simulated distribution')
    plt.title(var + ' - original distribution vs sumulated')
    plt.savefig(base_path + 'images/distribution_hist_real_simu_' + var + '.png')
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
# Compare the datasets CDF (real biomarkers vs virtual biomarkers)
for var in df_to_pred.columns:
    sns.ecdfplot(df_test[var], label='original dis')
    sns.ecdfplot(df_simu[var], label='simu dis')
    plt.title('CDF of ' + var + ' - original distribution vs sumulated')
    plt.savefig(base_path + 'images/CDF_hist_real_simu_' + var + '.png')
    plt.legend()
    plt.grid()
    plt.show()